# bpe_openai_gpt2

In [2]:
import os
import json
import regex as re # `regex` 是第三方库，不是 Python 自带的 `re` 库
import requests
from tqdm import tqdm
from functools import lru_cache 


> 1. `regex` 是第三方库，不是 Python 自带的 `re` 库
> 2. `lru` : Least Recently Used;  
> `functools.lru_cache()`: 是一个装饰器工厂，接收配置参数，返回一个装饰器，然后再用这个装饰器去处理函数；返回的装饰器给函数加一个“最近最少使用”缓存（LRU cache），把函数的返回值按参数缓存起来；以后用相同参数调用时，直接返回缓存结果，不再重新执行函数。

In [ ]:
@lru_cache()
def bytes_to_unicode():
    """
    Returns list of utf-8 byte and a corresponding list of unicode strings. 
    The reversible bpe codes work on unicode strings.
    This means you need a large #(number) of unicode characters in your vocab if you want to avoid UNKs.
    When you're at something like a 10B token dataset you end up needing around 5K for decent coverage.
    This is a significant percentage of your normal, say, 32K bpe vocab.
    To avoid that, we want lookup tables between utf-8 bytes and unicode strings.
    And avoids mapping to whitespace/control characters the bpe code barfs on.
    """
    """
    翻译：
    返回一个 utf-8 字节列表，以及与之一一对应的 unicode 字符串列表。
    可逆的 BPE 编码作用于 Unicode 字符串。
    这意味着：如果你想避免出现 UNKs（未知 token ），你的词表里需要包含大量的 Unicode 字符。
    当你处理规模约 100 亿（10B）token 的数据集时，你最终大约需要 5000 个（Unicode 字符）才能获得不错的覆盖率。
    对于常见的 32K 大小的 BPE 词表来说，5000 个 Unicode 字符会占掉相当大的比例。
    为了避免这种情况，我们希望在 UTF-8 字节和 Unicode 字符串之间建立查找表
    并且避免映射到空白字符/控制字符，因为 BPE 编码处理这些字符时会出错。
    """

    bs = list(range(ord("!"), ord("~") + 1)) + list(range(ord("¡"), ord("¬") + 1)) + list(range(ord("®"), ord("ÿ") + 1))
    cs = bs[:] # 安全字节 --> unicode 码点
    n = 0
    for b in range(2**8): # 危险字节 --> unicode 码点
        if b not in bs:
            bs.append(b)
            cs.append(2**8 + n)
            n += 1
    cs = [chr(n) for n in cs] # unicode 码点 --> unicode 字符
    return dict(zip(bs, cs)) # 最终实现：byte(0 ~ 255) --> unicode 字符

> 1. `ord()`: 返回单个字符对应的 Unicode 码点（十进制整数）
> 2. `chr()`: 把十进制整数（Unicode 码点）转换成对应的单个字符
> 3. `bs`: bytes source, UTF-8 字节的数值（0～255），作为 `bytes_to_unicode()` 方法返回的字典的 `key`
> 4. `cs`: codes destination, 目标 Unicode 码点的整数，之后会用 `chr()` 转成字符，作为 `bytes_to_unicode()` 方法返回的字典的 `value`

In [4]:
def get_pairs(word):
    """
    Return set of symbol pairs in a word.
    Word is represented as tuple of symbols (symbols being variable-length strings).
    """
    """
    翻译：
    返回一个`单词`(tuple)内所有相邻符号对的集合（set）
    `单词`用符号元组表示（每一个符号可以是可变长度的字符串）
    """
    pairs = set()
    prev_char = word[0]
    for char in word[1:]:
        pairs.add((prev_char, char))
        prev_char = char
    return pairs

> 1. 如何理解`"""Return set..."""` 中的内容?
>> - `单词`：不是指英文单词（如 `hello` ），是指一段被切分开的符号序列（如`("h", "e", "l", "l", "o")`）
>> - `可变长度`：符号可以是 `"a"`（1 个字符），也可以是 `"el"`、`"ing"`、`"hello"`

In [ ]:
class Encoder:
    def __init__(self, encoder, bpe_merges, errors="replace"):
        self.encoder = encoder # vocab(token --> token id)
        self.decoder = {v: k for k, v in self.encoder.items()}
        self.errors = errors  # how to handle errors in decoding
        self.byte_encoder = bytes_to_unicode() # byte(0 ~ 255) --> unicode 字符
        self.byte_decoder = {v: k for k, v in self.byte_encoder.items()}
        self.bpe_ranks = dict(zip(bpe_merges, range(len(bpe_merges))))
        self.cache = {}

        # Should have added re.IGNORECASE so BPE merges can happen for capitalized versions of contractions
        # contractions 指的是带撇号的英语缩略形式（如 `let's` 中的 `'s` ）
        self.pat = re.compile(r"""'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+""")

    def bpe(self, token):
        if token in self.cache: # 命中缓存，直接返回，跳过合并
            return self.cache[token]
        word = tuple(token) # unicode 字符串 --> unicode 字符序列
        pairs = get_pairs(word)

        if not pairs:
            return token

        while True:
            # 从 `pairs` 所有相邻符号对中，选出“优先级最高”的 pair
            bigram = min(pairs, key=lambda pair: self.bpe_ranks.get(pair, float("inf")))
            if bigram not in self.bpe_ranks:
                break
            # 合并 `word` 
            first, second = bigram
            new_word = []
            i = 0
            while i < len(word):
                try:
                    j = word.index(first, i) # 从位置 `i` 开始，查找 `first` 第一次出现的位置，返回下标 `j`
                    new_word.extend(word[i:j]) # 将 `word` 中从位置 `i` 到 `j`（不含 `j`）之间的所有符号，追加到 `new_word` 列表末尾
                    i = j
                except ValueError:
                    new_word.extend(word[i:])
                    break

                if word[i] == first and i < len(word) - 1 and word[i + 1] == second:
                    new_word.append(first + second)
                    i += 2
                else:
                    new_word.append(word[i])
                    i += 1
            new_word = tuple(new_word)
            word = new_word
            if len(word) == 1:
                break
            else:
                pairs = get_pairs(word)
        word = " ".join(word) # 把合并后的 `word`（符号序列）拼成一个用空格分隔的字符串
        self.cache[token] = word
        return word

    def encode(self, text):
        bpe_tokens = []
        for token in re.findall(self.pat, text):
            token = "".join(self.byte_encoder[b] for b in token.encode("utf-8"))
            bpe_tokens.extend(self.encoder[bpe_token] for bpe_token in self.bpe(token).split(" "))
        return bpe_tokens

    def decode(self, tokens):
        text = "".join([self.decoder[token] for token in tokens])
        text = bytearray([self.byte_decoder[c] for c in text]).decode("utf-8", errors=self.errors)
        return text

> 1. `bpe_merges`: bpe 算法的合并规则，一般存储在 vocab.bpe 中
> 2. `self.cache = {}`: 缓存, 同一个原始 token 多次进来，BPE 合并结果直接返回，不用重复跑合并循环，加速
> 3. `bpe()` 方法中传入的参数 `token` 指的是 raw text 片段经过 `UTF-8` 和 `byte_encoder` 处理后得到的 unicode 字符串
> 4. 理解 `bigram = min(pairs, key=lambda pair: self.bpe_ranks.get(pair, float("inf")))`：
>> - `bpe_ranks`: 数据类型为 dict, 里面储存 vocab.bpe 中的 bpe 合并规则。`key` 为合并规则，即一个相邻符号对；`value` 为排名，数字越小表示优先级越高
>> - `self.bpe_ranks.get(pair, float("inf"))`: 如果 `pair` 存在，返回对应的 value 值；如果不存在，返回 `float("inf")`
>> - `lambda pair: self.bpe_ranks.get(pair, float("inf"))` : 匿名函数，接受一个 `pair`, 返回它的 rank 或者 `float("inf")`
>> - `min(pairs, key=func)`: 遍历 `pairs` 中的每个元素，用 `func` 得到一个“比较值”，返回“比较值”最小的那个元素（pair）。`func`为上面提到的匿名函数